# Module 19: Parallel Processing

**Lesson: Threading, Multiprocessing, and AsyncIO for ML**

This notebook covers threading vs multiprocessing vs asyncio, the GIL, concurrent.futures, multiprocessing pools, shared memory, async/await basics, and choosing the right approach for ML workloads.

In [ ]:
import time
import numpy as np
import sys
print('Python version:', sys.version)
print('Ready to explore parallel processing')

## 1. Threading Basics and the GIL

Threads share memory but are limited by the Global Interpreter Lock (GIL). The GIL allows only one thread to execute Python bytecode at a time, making threads unsuitable for CPU-bound pure Python code. However, threads work well for IO-bound tasks where the thread releases the GIL while waiting.

In [ ]:
import threading
import time

# IO-bound task: threads work well here
def io_bound_task(task_id, duration):
    """Simulates an IO-bound operation like network request or file read"""
    time.sleep(duration)  # Thread releases GIL during sleep
    return f'Task {task_id} completed'

# CPU-bound task: threads do NOT help here
def cpu_bound_task(n):
    """Pure Python CPU-bound operation"""
    total = 0
    for i in range(n):
        total += i * i
    return total

# Demonstrate IO-bound with threading
start = time.time()
threads = []
for i in range(4):
    t = threading.Thread(target=io_bound_task, args=(i, 0.5))
    t.start()
    threads.append(t)
for t in threads:
    t.join()
io_time = time.time() - start

print('=== IO-bound with Threading ===')
print(f'4 x 0.5s IO tasks completed in {io_time:.2f}s')
print(f'Sequential would take: {4 * 0.5:.1f}s')
print(f'Speedup: {4 * 0.5 / io_time:.1f}x')
print('Threading excels at IO-bound work because threads release GIL during waits')

In [ ]:
# Demonstrate GIL limitation on CPU-bound tasks with threads
def count_heavy(n):
    total = 0
    for i in range(n):
        total += i ** 2 + i ** 3
    return total

# Sequential
start = time.time()
results_seq = [count_heavy(5000000) for _ in range(4)]
seq_time = time.time() - start

# Threading
start = time.time()
threads = []
results = []
def worker(n):
    results.append(count_heavy(n))
for _ in range(4):
    t = threading.Thread(target=worker, args=(5000000,))
    t.start()
    threads.append(t)
for t in threads:
    t.join()
thread_time = time.time() - start

print('=== GIL Impact on CPU-bound Tasks ===')
print(f'Sequential: {seq_time:.2f}s')
print(f'Threading:  {thread_time:.2f}s')
print(f'Ratio: {thread_time / seq_time:.2f}x (should be ~1.0 due to GIL)')
print('Threading does NOT speed up CPU-bound pure Python code due to the GIL')

## 2. Multiprocessing for CPU-bound Tasks

Multiprocessing creates separate processes, each with its own GIL, enabling true parallelism. Use it for CPU-bound ML tasks.

In [ ]:
from multiprocessing import Pool, cpu_count

print(f'CPU cores available: {cpu_count()}')

# CPU-bound with multiprocessing
start = time.time()
with Pool(processes=4) as pool:
    mp_results = pool.map(count_heavy, [5000000] * 4)
mp_time = time.time() - start

print(f'\n=== CPU-bound with Multiprocessing ===')
print(f'Sequential:      {seq_time:.2f}s')
print(f'Threading:       {thread_time:.2f}s (GIL bound)')
print(f'Multiprocessing: {mp_time:.2f}s (true parallelism)')
print(f'Speedup vs seq: {seq_time / mp_time:.1f}x')

print('\nKey insight: Multiprocessing bypasses the GIL with separate processes')

## 3. concurrent.futures: High-level API

concurrent.futures provides a simple, consistent interface for both threading and multiprocessing.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ThreadPoolExecutor for IO-bound
def fetch_simulated_url(url_id):
    time.sleep(0.2)  # Simulate network
    return f'Data from endpoint {url_id}'

urls = list(range(8))

print('=== ThreadPoolExecutor (IO-bound) ===')
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(fetch_simulated_url, u) for u in urls]
    for future in as_completed(futures):
        print(f'  Got: {future.result()}')
print(f'Time: {time.time() - start:.2f}s for 8 tasks')

print('\n=== ProcessPoolExecutor (CPU-bound) ===')
def heavy_compute(x):
    return sum(i ** 2 for i in range(x * 1000000))

start = time.time()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(heavy_compute, [1, 2, 3, 4]))
print(f'Results: {results}')
print(f'Time: {time.time() - start:.2f}s')

## 4. Multiprocessing Pool and Shared Memory

The multiprocessing module provides Pool for managing worker processes and mechanisms for sharing data between processes.

In [ ]:
from multiprocessing import Pool, Value, Array, Manager

# Multiprocessing Pool variations
def square(x):
    return x * x

with Pool(processes=4) as pool:
    # map: block until all results ready
    results_map = pool.map(square, range(10))
    print('map:', results_map)
    
    # imap: lazy iteration (returns results as they come)
    results_imap = list(pool.imap(square, range(10)))
    print('imap:', results_imap)
    
    # starmap: multiple arguments
    def multiply(a, b):
        return a * b
    results_star = pool.starmap(multiply, [(1,2), (3,4), (5,6)])
    print('starmap:', results_star)
    
    # apply_async: non-blocking with callback
    results_async = [pool.apply_async(square, (i,)) for i in range(5)]
    print('apply_async:', [r.get() for r in results_async])

# Shared memory with Manager
print('\n=== Shared State with Manager ===')
with Manager() as manager:
    shared_dict = manager.dict()
    shared_list = manager.list()
    
    def worker_proc(key, value):
        shared_dict[key] = value
        shared_list.append(value)
    
    with Pool(processes=2) as pool:
        pool.starmap(worker_proc, [('a', 1), ('b', 2)])
    
    print('Shared dict:', dict(shared_dict))
    print('Shared list:', list(shared_list))

## 5. AsyncIO Fundamentals

asyncio uses cooperative multitasking within a single thread. It's ideal for IO-bound tasks with many concurrent connections.

In [ ]:
import asyncio

async def async_fetch(endpoint_id):
    """Simulate an async API call"""
    await asyncio.sleep(0.2)  # Non-blocking sleep
    return {'endpoint': endpoint_id, 'data': f'result_{endpoint_id}'}

async def demo_async():
    start = time.time()
    
    # Sequential async
    results_seq = []
    for i in range(5):
        result = await async_fetch(i)
        results_seq.append(result)
    t_seq = time.time() - start
    
    # Concurrent async with gather
    start = time.time()
    tasks = [async_fetch(i) for i in range(5)]
    results_concurrent = await asyncio.gather(*tasks)
    t_concurrent = time.time() - start
    
    print('=== AsyncIO ===')
    print(f'Sequential async: {t_seq:.3f}s')
    print(f'Concurrent gather: {t_concurrent:.3f}s')
    print(f'Speedup: {t_seq / t_concurrent:.1f}x')
    print(f'Results: {[r["data"] for r in results_concurrent]}')
    print('\nasync/await is perfect for high-concurrency IO workloads (web servers, API clients)')

asyncio.run(demo_async())

## 6. ML Focus: Parallelizing Data Preprocessing

Data preprocessing is often embarassingly parallel — each row or chunk can be processed independently.

In [ ]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

# Generate large dataset
np.random.seed(42)
n_rows = 100000
df_large = pd.DataFrame({
    'feature_1': np.random.randn(n_rows),
    'feature_2': np.random.randn(n_rows),
    'feature_3': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows),
    'feature_4': np.random.randn(n_rows) * 10 + 50,
    'feature_5': np.random.exponential(scale=2, size=n_rows),
    'target': np.random.randint(0, 2, n_rows)
})
# Add some missing values
df_large.loc[np.random.choice(n_rows, 1000), 'feature_1'] = np.nan
df_large.loc[np.random.choice(n_rows, 500), 'feature_4'] = np.nan

print(f'Dataset: {len(df_large):,} rows, {len(df_large.columns)} columns')
print(f'Missing values: {df_large.isna().sum().sum()}')

# Sequential preprocessing
def preprocess(df):
    df = df.copy()
    # Impute
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col].fillna(df[col].median(), inplace=True)
    # Clip outliers
    for col in df.select_dtypes(include=[np.number]).columns:
        mean, std = df[col].mean(), df[col].std()
        df[col] = df[col].clip(mean - 3*std, mean + 3*std)
    return df

start = time.time()
result_seq = preprocess(df_large)
seq_time = time.time() - start
print(f'\nSequential preprocessing: {seq_time:.3f}s')

In [ ]:
# Parallel preprocessing with chunking
def preprocess_chunk(chunk_tuple):
    idx, chunk = chunk_tuple
    return preprocess(chunk)

n_chunks = 4
chunks = [(i, df_large.iloc[i::n_chunks].copy()) for i in range(n_chunks)]

start = time.time()
with ProcessPoolExecutor(max_workers=n_chunks) as executor:
    processed_chunks = list(executor.map(preprocess_chunk, chunks))

result_parallel = pd.concat(processed_chunks).sort_index()
parallel_time = time.time() - start

print('=== Parallel Preprocessing ===')
print(f'Chunks: {n_chunks}')
print(f'Parallel time: {parallel_time:.3f}s')
print(f'Sequential time: {seq_time:.3f}s')
print(f'Speedup: {seq_time / parallel_time:.2f}x')
print(f'Data integrity: {len(result_parallel)} rows, {result_parallel.isna().sum().sum()} missing values')

print('\nTip: sklearn's n_jobs=-1 is often simpler than manual chunking')

## 7. ML Focus: Parallel Model Training and Inference

Many ML libraries (sklearn, xgboost) handle parallelism internally. For custom training loops or ensemble methods, use multiprocessing.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# Generate data
X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=42)

# sklearn parallel training (n_jobs=-1 uses all cores)
print('=== sklearn Internal Parallelism ===')
print('Many sklearn estimators support n_jobs=-1 for parallelism')

start = time.time()
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X, y)
rf_time = time.time() - start
print(f'Random Forest (n_jobs=-1): {rf_time:.3f}s')

start = time.time()
lr = LogisticRegression(n_jobs=-1)
lr.fit(X, y)
lr_time = time.time() - start
print(f'Logistic Regression (n_jobs=-1): {lr_time:.3f}s')

# Parallel training of multiple models
print('\n=== Parallel Model Training ===')
def train_and_score(model_class, params, X_train, y_train, X_test, y_test):
    model = model_class(**params)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return model_class.__name__, score

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_configs = [
    (RandomForestClassifier, {'n_estimators': 100, 'n_jobs': 1, 'random_state': 42}),
    (GradientBoostingClassifier, {'n_estimators': 50, 'random_state': 42}),
    (LogisticRegression, {'max_iter': 1000, 'n_jobs': 1}),
]

start = time.time()
with ProcessPoolExecutor(max_workers=3) as executor:
    futures = [
        executor.submit(train_and_score, cls, params, X_train, y_train, X_test, y_test)
        for cls, params in model_configs
    ]
    for future in as_completed(futures):
        name, score = future.result()
        print(f'  {name}: accuracy = {score:.4f}')
print(f'Parallel training time: {time.time() - start:.3f}s')

## 8. When to Use Each Approach

A decision framework for choosing the right concurrency model.

In [ ]:
print('' + '='*60)
print('DECISION FRAMEWORK: Choosing Parallelism Strategy')
print('='*60)
print('''
Is the task IO-bound or CPU-bound?
|
+-- IO-bound (network, disk, API calls):
|   |
|   +-- Many concurrent connections?  --> asyncio
|   |                                      (fastest, lowest overhead)
|   |
|   +-- Few connections, blocking libs? --> ThreadPoolExecutor
|                                          (simpler than asyncio)
|
+-- CPU-bound (computation, training):
    |
    +-- Pure Python code?           --> multiprocessing / ProcessPoolExecutor
    |                                   (bypasses GIL)
    |
    +-- Uses NumPy/scikit-learn?    --> Let library handle it (n_jobs=-1)
    |                                   (C extensions release GIL)
    |
    +-- Custom ML training loop?    --> ProcessPoolExecutor or joblib
'')

print('--- Summary ---')
print('threading:       IO-bound, shared memory, GIL-limited for CPU')
print('multiprocessing: CPU-bound, true parallelism, separate memory')
print('asyncio:         Many IO connections, single-thread, cooperative')
print('concurrent.futures: High-level API for TPE and PPE')

## Summary

In this lesson, you learned:
- Threading is best for IO-bound tasks but limited by GIL for CPU-bound work
- Multiprocessing enables true parallelism for CPU-bound ML tasks
- concurrent.futures provides a clean API for both thread and process pools
- AsyncIO excels at high-concurrency IO workloads
- The GIL only affects pure Python CPU-bound code (C extensions release it)
- sklearn's n_jobs=-1 handles most ML parallelism automatically
- For custom parallel pipelines, use ProcessPoolExecutor with chunking

**Golden Rule:** Profile first, then optimize. Start sequential, identify bottlenecks, then parallelize.